In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.common.exceptions import WebDriverException
from webdriver_manager.chrome import ChromeDriverManager
import concurrent.futures
import time

DEBUG_PORT = 9222

def try_connect_existing_chrome():
    options = webdriver.ChromeOptions()
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option(
        "debuggerAddress", f"127.0.0.1:{DEBUG_PORT}"
    )
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)


def get_or_create_driver(timeout=5):
    start_time = time.time()
    while time.time() - start_time < timeout:
        try:
            print("🔁 Tentative de connexion à Chrome existant...")
            with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
                future = executor.submit(try_connect_existing_chrome)
                driver = future.result(timeout=timeout)
            print("✅ Connecté à Chrome existant")
            return driver
        except (WebDriverException, concurrent.futures.TimeoutError):
            print("⏳ Chrome non dispo ou timeout, retry...")
            time.sleep(0.5)

    # Après timeout → lancement d'un nouveau Chrome
    print("🚀 Timeout atteint → lancement d'un nouveau Chrome")
    options = webdriver.ChromeOptions()
    options.add_argument(f"--remote-debugging-port={DEBUG_PORT}")
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    print("🆕 Nouveau Chrome lancé avec debugging")
    return driver


In [ ]:
# driver = get_or_create_driver()
# driver.get("https://dpm.lol/tierlist?tier=gold_plus")

🔁 Tentative de connexion à Chrome existant...
⏳ Chrome non dispo ou timeout, retry...
🚀 Timeout atteint → lancement d'un nouveau Chrome
🆕 Nouveau Chrome lancé avec debugging


In [3]:
def get_last_radix_buttons():
    """
    Récupère les boutons du dernier pop-up Radix ouvert.
    Utilise l'ID commençant par 'radix-' pour identifier le pop-up correct.
    """

    # 1️⃣ chercher tous les éléments dont l'ID commence par 'radix-'
    radix_roots = driver.find_elements(By.XPATH, "//*[starts-with(@id, 'radix-')]")
    if not radix_roots:

        return []

    # 2️⃣ prendre le dernier pop-up (le plus récemment ouvert)
    radix_root = radix_roots[-1]

    # 3️⃣ le div interne qui contient les boutons
    try:
        radix_div = radix_root.find_element(By.XPATH, "./div")
    except:

        return []

    # 4️⃣ récupérer les boutons
    buttons = radix_div.find_elements(By.TAG_NAME, "button")

    return buttons

In [4]:
# def init_parse(driver, scroll_pause=1.0, scroll_step=500):

#     champions_by_key = {}

#     scroll_top = 0
#     print("🚀 init_parse() démarré")

#     while True:
#         driver.execute_script("window.scrollTo(0, arguments[0]);", scroll_top)
#         time.sleep(scroll_pause)

#         container_selector = (
#             "#root > main > div > div.flex.justify-center.gap-16 > "
#             "div.flex.flex-col.max-w-5xl.w-full.items-center.md\\:items-start.my-8.gap-8.px-12.lg\\:px-0 > "
#             "div.flex.flex-col.w-full.gap-12.bg-black-800.border.border-black-0\\/10.px-16.py-12.rounded-md > "
#             "div.flex.flex-col.gap-y-8.w-full.mb-48 > div:nth-child(3) > div"
#         )

#         container = driver.find_element(By.CSS_SELECTOR, container_selector)
#         rows = container.find_elements(By.XPATH, "./div/div")

#         for idx, row in enumerate(rows):
#             text = row.text.strip()
#             if not text:
#                 continue

#             lines = text.splitlines()
#             label = lines[0]
#             name = lines[1] if len(lines) > 1 else "Unknown"

#             key = text  # ou (label, name)

#             champions_by_key[key] = {
#                 "scroll_y": scroll_top,
#                 "label": label,
#                 "numero": idx,
#                 "name": name,
#             }

#         scroll_top += scroll_step
#         new_height = driver.execute_script("return document.body.scrollHeight")

#         if scroll_top >= new_height:
#             break

#     champions = list(champions_by_key.values())

#     print(f"✅ {len(champions)} champions collectés (dernières occurrences)")
#     return champions


In [5]:
# import time
# from selenium.webdriver.common.by import By

# def init_parse(driver, scroll_pause=1.0, scroll_step=500):

#     champions_by_key = {}

#     scroll_top = 0
#     print("🚀 init_parse() démarré")

#     container_selector = (
#         "#root > main > div > div.flex.justify-center.gap-16 > "
#         "div.flex.flex-col.max-w-5xl.w-full.items-center.md\\:items-start.my-8.gap-8.px-12.lg\\:px-0 > "
#         "div.flex.flex-col.w-full.gap-12.bg-black-800.border.border-black-0\\/10.px-16.py-12.rounded-md > "
#         "div.flex.flex-col.gap-y-8.w-full.mb-48 > div:nth-child(3) > div"
#     )

#     while True:
#         # scroll page
#         driver.execute_script("window.scrollTo(0, arguments[0]);", scroll_top)
#         time.sleep(scroll_pause)

#         container = driver.find_element(By.CSS_SELECTOR, container_selector)

#         # 📌 lire positions réelles
#         page_scroll_y = driver.execute_script("return window.pageYOffset;")
#         table_scroll_y = driver.execute_script(
#             "return arguments[0].scrollTop;", container
#         )

#         rows = container.find_elements(By.XPATH, "./div/div")

#         for idx, row in enumerate(rows):
#             text = row.text.strip()
#             if not text:
#                 continue

#             lines = text.splitlines()
#             label = lines[0]
#             name = lines[1] if len(lines) > 1 else "Unknown"

#             key = text

#             if key not in champions_by_key:
#                 champions_by_key[key] = {
#                     "scroll_positions": [],
#                     "label": label,
#                     "numero": idx,
#                     "name": name,
#                 }

#             champions_by_key[key]["scroll_positions"].append({
#                 "page": page_scroll_y,
#                 "table": table_scroll_y
#             })

#         scroll_top += scroll_step
#         new_height = driver.execute_script("return document.body.scrollHeight")

#         if scroll_top >= new_height:
#             break

#     # 🔁 position finale = moyenne des 2 dernières occurrences
#     champions = []

#     for champ in champions_by_key.values():
#         positions = champ["scroll_positions"]

#         if len(positions) >= 2:
#             p1, p2 = positions[-2], positions[-1]
#             champ["scroll_page"] = (p1["page"] + p2["page"]) // 2
#             champ["scroll_table"] = (p1["table"] + p2["table"]) // 2
#         else:
#             champ["scroll_page"] = positions[-1]["page"]
#             champ["scroll_table"] = positions[-1]["table"]

#         del champ["scroll_positions"]
#         champions.append(champ)

#     print(f"✅ {len(champions)} champions collectés (scroll page + table)")
#     return champions


In [6]:
import time
from selenium.webdriver.common.by import By

def init_parse(driver, scroll_pause=1.0, scroll_step=500):

    champions_by_key = {}

    scroll_top = 0
    print("🚀 init_parse() démarré")

    container_selector = (
        "#root > main > div > div.flex.justify-center.gap-16 > "
        "div.flex.flex-col.max-w-5xl.w-full.items-center.md\\:items-start.my-8.gap-8.px-12.lg\\:px-0 > "
        "div.flex.flex-col.w-full.gap-12.bg-black-800.border.border-black-0\\/10.px-16.py-12.rounded-md > "
        "div.flex.flex-col.gap-y-8.w-full.mb-48 > div:nth-child(3) > div"
    )

    while True:
        driver.execute_script("window.scrollTo(0, arguments[0]);", scroll_top)
        time.sleep(scroll_pause)

        container = driver.find_element(By.CSS_SELECTOR, container_selector)

        page_scroll_y = driver.execute_script("return window.pageYOffset;")
        table_scroll_y = driver.execute_script(
            "return arguments[0].scrollTop;", container
        )

        rows = container.find_elements(By.XPATH, "./div/div")

        for idx, row in enumerate(rows):
            text = row.text.strip()
            if not text:
                continue

            lines = text.splitlines()
            label = lines[0]
            name = lines[1] if len(lines) > 1 else "Unknown"

            # ✅ récupération URL
            url = None
            try:
                link = row.find_element(By.XPATH, ".//a")
                url = link.get_attribute("href")
            except:
                pass

            if url:
                print(f"🔗 {label} → {url}")

            key = text

            if key not in champions_by_key:
                champions_by_key[key] = {
                    "scroll_positions": [],
                    "label": label,
                    "numero": idx,
                    "name": name,
                    "url": url,   # ✅ stockée ici
                }

            champions_by_key[key]["scroll_positions"].append({
                "page": page_scroll_y,
                "table": table_scroll_y
            })

        scroll_top += scroll_step
        new_height = driver.execute_script("return document.body.scrollHeight")

        if scroll_top >= new_height:
            break

    champions = []

    for champ in champions_by_key.values():
        positions = champ["scroll_positions"]

        if len(positions) >= 2:
            p1, p2 = positions[-2], positions[-1]
            champ["scroll_page"] = (p1["page"] + p2["page"]) // 2
            champ["scroll_table"] = (p1["table"] + p2["table"]) // 2
        else:
            champ["scroll_page"] = positions[-1]["page"]
            champ["scroll_table"] = positions[-1]["table"]

        del champ["scroll_positions"]
        champions.append(champ)

    print(f"✅ {len(champions)} champions collectés (scroll + url)")
    return champions


In [7]:
import hashlib
from selenium.webdriver.common.by import By

ROLE_HASH_TO_TEXT = {
    "df14d23b35c9842bd8afa0db2b922aaf": "top",
    "bd9e54c883010f9bc2487e7d26a91b77": "jun",
    "3ce111209b6d69bee8498e94b02567ad": "mid",
    "6f1d8859e29002c2c45b527544e5a755": "adc",
    "3b66426a9b2beeca218cc726985f68b1": "sup",
}

def resolve_role_from_hash(svg_hash: str) -> str:
    role = ROLE_HASH_TO_TEXT.get(svg_hash)

    if role is None:
        print(f"⚠️ Hash de rôle inconnu : {svg_hash}")
        return "unknown"

    return role

# svg_hash = hashlib.md5(role_svg.encode("utf-8")).hexdigest()

def hash_svg_path(svg_element) -> str:
    """
    Extrait le path 'd' du SVG et retourne son hash MD5
    """
    paths = svg_element.find_elements(By.TAG_NAME, "path")
    if not paths:
        return None

    path_d = paths[0].get_attribute("d").strip()
    return hashlib.md5(path_d.encode("utf-8")).hexdigest()


In [8]:
def get_selected_played_lane(driver) -> dict:
    print("🟢 Détection de la played lane sélectionnée")

    # 1️⃣ on récupère TOUS les containers possibles (sécurise si la page évolue)
    containers = driver.find_elements(
        By.CSS_SELECTOR,
        "div.border-black-600.flex.w-fit.items-center.justify-center"
    )

    print(f"🔍 {len(containers)} containers candidats trouvés")

    if not containers:
        print("❌ Aucun container trouvé")
        return {"lane": None, "percentage": None}

    # 2️⃣ on prend le premier (structure unique sur la page)
    container = containers[0]

    lane_divs = container.find_elements(By.XPATH, "./div")
    print(f"🔍 {len(lane_divs)} boutons de lane trouvés")

    # 3️⃣ on parcourt uniquement les 5 boutons
    for idx, lane_div in enumerate(lane_divs):
        try:
            link = lane_div.find_element(By.TAG_NAME, "a")
            inner_div = link.find_element(By.TAG_NAME, "div")

            class_name = inner_div.get_attribute("class") or ""
            print(f"[Lane {idx}] classes = {class_name}")

            # pas sélectionné → on skip
            if "bg-blue-200" not in class_name:
                continue

            print(f"⭐ Lane sélectionnée détectée (index {idx})")

            # SVG → hash → lane
            svg = inner_div.find_element(By.TAG_NAME, "svg")
            svg_hash = hash_svg_path(svg)
            lane = resolve_role_from_hash(svg_hash)

            print(f"   🔐 svg_hash = {svg_hash}")
            print(f"   🏷️ lane     = {lane}")

            # 4️⃣ span JUSTE APRÈS le <a>
            percentage = None
            try:
                span = lane_div.find_element(By.TAG_NAME, "span")
                percentage = span.text.strip()
            except Exception:
                print("⚠️ Span pourcentage introuvable")

            print(f"   📊 percentage = {percentage}")

            return {
                "lane": lane,
                "percentage": percentage
            }

        except Exception as e:
            print(f"[Lane {idx}] ❌ erreur : {e}")

    print("❌ Aucune lane sélectionnée trouvée")
    return {"lane": None, "percentage": None}


In [9]:
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException

def collect_champion_lane_stats(driver) -> dict:
    print("🟢 Collecte des stats champion / lane")

    stats = {
        "tier": None,
        "rank": None,
        "winrate": None,
        "pickrate": None,
        "banrate": None,
        "games": None,
    }

    try:
        container = driver.find_element(
            By.XPATH,
            '//*[@id="root"]/main/div/div[3]/div[2]/div/div[2]/div[1]'
        )
    except NoSuchElementException:
        print("❌ Container stats introuvable")
        return stats

    print("✅ Container stats trouvé")
    print("🎨 class :", container.get_attribute("class"))

    spans = container.find_elements(By.TAG_NAME, "span")
    print(f"🔢 {len(spans)} spans trouvés\n")

    KNOWN_KEYS = {
        "niveau": "tier",
        "rang": "rank",
        "winrate": "winrate",
        "pickrate": "pickrate",
        "banrate": "banrate",
        "parties": "games",
    }

    for idx, span in enumerate(spans):
        raw_text = span.text.strip()
        if not raw_text:
            continue

        print(f"[Span {idx}] → '{raw_text}'")

        lower = raw_text.lower()

        for label, key in KNOWN_KEYS.items():
            if label in lower:
                # ne pas écraser une valeur déjà trouvée
                if stats[key] is not None:
                    continue

                # extraction valeur
                value = (
                    raw_text
                    .replace(label, "")
                    .replace("\n", " ")
                    .strip()
                )

                # ignorer les spans "label only"
                if not value:
                    continue

                # suppression de l'espace insécable pour "games"
                if key == "games":
                    value = value.replace("\u202f", "").replace(" ", "")

                stats[key] = value
                print(f"✅ {key} détecté → '{value}'")

    print("-" * 60)
    print("✅ Stats collectées :", stats)
    return stats


In [10]:
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException, WebDriverException
import time

def click_matchup_or_synergy(driver, matchup: bool) -> bool:
    target_name = "MATCHUPS" if matchup else "SYNERGIES"
    aria_key = "Enemy" if matchup else "Ally"

    print(f"🟢 Sélection de l'onglet {target_name}", flush=True)

    xpath = f"//button[@role='tab' and contains(@aria-controls, '{aria_key}')]"

    try:
        button = driver.find_element(By.XPATH, xpath)

        data_state = button.get_attribute("data-state")
        print(f"🔍 {target_name} data-state = {data_state}")

        if data_state == "active":
            print(f"ℹ️ Onglet {target_name} déjà actif")
            return False

        driver.execute_script(
            "arguments[0].scrollIntoView({block: 'center'});",
            button
        )
        time.sleep(0.3)

        button.click()
        print(f"🖱️ Onglet {target_name} cliqué", flush=True)
        time.sleep(0.7)

        return True

    except NoSuchElementException:
        print(f"❌ Onglet {target_name} introuvable (aria-controls)", flush=True)
        return False

    except WebDriverException as e:
        print(f"❌ Erreur Selenium sur {target_name} : {e}", flush=True)
        return False


In [11]:
def click_matchup_or_synergy_lane(driver, lane_name) -> bool:
    print(f"🟢 Sélection de la lane {lane_name} via hash du PATH SVG")

    time.sleep(0.5)

    lane_container_xpath = (
        '//*[@id="root"]/main[1]/div[1]/div[3]/div[2]/div[1]/div[2]/div[4]'
        '/div[1]/div[1]/div[3]/div[1]'
    )

    try:
        lane_container = driver.find_element(By.XPATH, lane_container_xpath)
    except Exception as e:
        print(f"❌ Conteneur des lanes introuvable : {e}")
        return False

    lane_buttons = lane_container.find_elements(By.TAG_NAME, "button")
    print(f"🔍 {len(lane_buttons)} boutons de lane trouvés\n")

    for i, btn in enumerate(lane_buttons):
        svgs = btn.find_elements(By.TAG_NAME, "svg")
        if not svgs:
            continue

        svg = svgs[0]
        svg_hash = hash_svg_path(svg)

        if not svg_hash:
            continue

        role = resolve_role_from_hash(svg_hash)

        print(f"[Lane {i}] hash={svg_hash} role={role}")

        if role == lane_name:
            print(f"🎯 Bouton {lane_name} identifié — clic")

            driver.execute_script(
                "arguments[0].scrollIntoView({block: 'center'});",
                btn,
            )
            time.sleep(0.3)
            btn.click()

            print(f"🖱️ Bouton {lane_name} cliqué avec succès")
            return True

    print(f"❌ Bouton {lane_name} non trouvé")
    return False


In [12]:
from selenium.webdriver.common.by import By
from selenium.common.exceptions import (
    NoSuchElementException,
    StaleElementReferenceException,
    ElementClickInterceptedException,
    ElementNotInteractableException,
)
import time


def click_liste_complete(driver) -> bool:
    print("🟢 Recherche du bouton '+ Liste complète' parmi tous les boutons...", flush=True)

    for attempt in range(2):
        print(f"🔁 Tentative {attempt + 1}/2")

        try:
            buttons = driver.find_elements(By.TAG_NAME, "button")
            print(f"🔍 {len(buttons)} boutons trouvés sur la page")

            for i, btn in enumerate(buttons):
                try:
                    btn_text = btn.text.strip()
                    btn_text_clean = " ".join(btn_text.split())

                    # XPath absolu (debug)
                    btn_xpath = driver.execute_script(
                        """
                        function getXPath(element) {
                            if (element.id !== '')
                                return '//*[@id="' + element.id + '"]';
                            if (element === document.body)
                                return '/html/body';

                            let ix = 0;
                            let siblings = element.parentNode.childNodes;
                            for (let i = 0; i < siblings.length; i++) {
                                let sibling = siblings[i];
                                if (sibling === element)
                                    return getXPath(element.parentNode) + '/' +
                                        element.tagName.toLowerCase() + '[' + (ix + 1) + ']';
                                if (sibling.nodeType === 1 && sibling.tagName === element.tagName)
                                    ix++;
                            }
                        }
                        return getXPath(arguments[0]);
                        """,
                        btn,
                    )

                    print(f"[{i}] → '{btn_text_clean}' | XPath: {btn_xpath}")

                    if "+ Liste complète" in btn_text_clean:
                        print("🎯 Bouton '+ Liste complète' détecté, tentative de clic...")

                        driver.execute_script(
                            "arguments[0].scrollIntoView({block: 'center'});",
                            btn,
                        )
                        time.sleep(0.75)
                        btn.click()

                        print("🖱️ Bouton '+ Liste complète' cliqué avec succès", flush=True)
                        time.sleep(1)
                        return True  # ✅ succès immédiat

                except StaleElementReferenceException:
                    print(f"⚠️ Bouton [{i}] devenu obsolète (DOM mis à jour)")
                except Exception as e:
                    print(f"⚠️ Erreur sur le bouton [{i}] : {e}")

        except (
            NoSuchElementException,
            ElementClickInterceptedException,
            ElementNotInteractableException,
        ) as e:
            print(f"❌ Erreur lors de la tentative {attempt + 1} : {e}")

        time.sleep(0.5)

    print("❌ Bouton '+ Liste complète' non cliqué après 2 tentatives", flush=True)
    return False  # ❌ échec final


In [13]:
# from selenium.webdriver.common.by import By
# from selenium.common.exceptions import NoSuchElementException, StaleElementReferenceException
# import time

# def collect_matchup(driver):
#     """
#     Parcourt les matchups Enemy après clic sur 'Liste complète'
#     et retourne une liste de dicts :
#     {
#         champ_counter_i,
#         winrate,
#         games,
#         lane_quality
#     }
#     """

#     print("🟢 Démarrage collect_matchup")
#     results = []
#     seen = set()

#     # ===============================
#     # 1️⃣ Section Enemy
#     # ===============================
#     # try:
#     #     enemy_root = driver.find_element(By.XPATH, '//*[@id="radix-_r_8_-content-Enemy"]')
#     #     print(f"✅ Section Enemy trouvée (ID: {enemy_root.get_attribute('id')})")
#     # except NoSuchElementException:
#     #     print("❌ Section Enemy introuvable !")
#     #     return results

#     enemy_root = None
#     for attempt in range(2):
#         try:
#             enemy_root = driver.find_element(By.XPATH, '//*[@id="radix-_r_8_-content-Enemy"]')
#             print(f"✅ Section Enemy trouvée (ID: {enemy_root.get_attribute('id')})")
#             break
#         except NoSuchElementException:
#             print(f"⏳ Tentative {attempt + 1}/2 : Section Enemy introuvable")
#             time.sleep(0.8)  # petit sleep (ajuste si besoin)

#     if enemy_root is None:
#         print("❌ Section Enemy introuvable après 2 tentatives !")
#         return results

#     # ===============================
#     # 2️⃣ Récupérer les deux wrappers : visible + hors écran
#     # ===============================
#     try:
#         visible_wrapper = enemy_root.find_element(By.XPATH, "./div/div[1]/div")
#         hidden_wrapper = enemy_root.find_element(By.XPATH, "./div/div[2]/div")
#         print("✅ Wrappers visible et hidden trouvés")
#     except NoSuchElementException:
#         print("❌ Wrappers introuvables !")
#         return results

#     # ===============================
#     # 3️⃣ Fonction de parsing des cards
#     # ===============================
#     def parse_cards(container):
#         cards = container.find_elements(By.XPATH, "./div")
#         print(f"🔍 {len(cards)} cards trouvées dans ce container")
#         for idx, card in enumerate(cards):
#             try:
#                 # ---------- <a> ----------
#                 anchor = card.find_element(By.XPATH, "./a")
#                 img = anchor.find_element(By.XPATH, ".//img")
#                 champ_name = img.get_attribute("alt").strip()
#                 if not champ_name or champ_name in seen:
#                     continue
#                 seen.add(champ_name)

#                 # -------- winrate et nombre de parties --------
#                 info_div = anchor.find_element(By.XPATH, "./div/div[2]")  # div C
#                 spans = info_div.find_elements(By.XPATH, "./span")
#                 winrate = spans[0].text.strip() if len(spans) > 0 else ""
#                 games = spans[1].text.strip() if len(spans) > 1 else ""

#                 # -------- lane_quality --------
#                 lane_quality = ""
#                 try:
#                     button = card.find_element(By.XPATH, "./button")
#                     lane_quality = button.text.strip()
#                 except NoSuchElementException:
#                     pass

#                 print(f"🎯 {champ_name} | Winrate: {winrate} | Games: {games} | Lane: {lane_quality}")

#                 results.append({
#                     "champ_counter_i": champ_name,
#                     "winrate": winrate,
#                     "games": games,
#                     "lane_quality": lane_quality
#                 })

#             except StaleElementReferenceException:
#                 print("⚠️ StaleElement — skip")
#             except Exception as e:
#                 print(f"❌ Erreur card {idx} → {e}")

#     # ===============================
#     # 4️⃣ Parser les cards visibles et hors écran
#     # ===============================
#     parse_cards(visible_wrapper)
#     parse_cards(hidden_wrapper)

#     print(f"📦 Matchups collectés : {len(results)}")
#     return results


In [14]:
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException, StaleElementReferenceException
import time

def collect_matchup(driver, matchup: bool = True):
    """
    Parcourt les matchups (Enemy) ou synergies (Ally) après clic sur 'Liste complète'
    et retourne une liste de dicts :
    {
        champ_counter_i,
        winrate,
        games,
        lane_quality
    }

    :param matchup: True pour matchups (Enemy), False pour synergies (Ally)
    """

    print("🟢 Démarrage collect_matchup")
    results = []
    seen = set()

    # ===============================
    # 1️⃣ Section Enemy / Ally
    # ===============================
    aria_key = "Enemy" if matchup else "Ally"
    container = None
    for attempt in range(2):
        try:
            # on sélectionne le container via role="tabpanel" et aria-labelledby
            container = driver.find_element(
                By.XPATH,
                f"//div[@role='tabpanel' and contains(@aria-labelledby, '{aria_key}')]"
            )
            print(f"✅ Section {'Enemy' if matchup else 'Ally'} trouvée (aria-labelledby: {aria_key})")
            break
        except NoSuchElementException:
            print(f"⏳ Tentative {attempt+1}/2 : Section {'Enemy' if matchup else 'Ally'} introuvable")
            time.sleep(0.8)

    if container is None:
        print(f"❌ Section {'Enemy' if matchup else 'Ally'} introuvable après 2 tentatives !")
        return results

    # ===============================
    # 2️⃣ Récupérer les deux wrappers : visible + hors écran
    # ===============================
    try:
        visible_wrapper = container.find_element(By.XPATH, "./div/div[1]/div")
        hidden_wrapper = container.find_element(By.XPATH, "./div/div[2]/div")
        print("✅ Wrappers visible et hidden trouvés")
    except NoSuchElementException:
        print("❌ Wrappers introuvables !")
        return results

    # ===============================
    # 3️⃣ Fonction de parsing des cards
    # ===============================
    def parse_cards(wrapper):
        cards = wrapper.find_elements(By.XPATH, "./div")
        print(f"🔍 {len(cards)} cards trouvées dans ce wrapper")
        for idx, card in enumerate(cards):
            try:
                # ---------- <a> ----------
                anchor = card.find_element(By.XPATH, "./a")
                img = anchor.find_element(By.XPATH, ".//img")
                champ_name = img.get_attribute("alt").strip()
                if not champ_name or champ_name in seen:
                    continue
                seen.add(champ_name)

                # -------- winrate et nombre de parties --------
                info_div = anchor.find_element(By.XPATH, "./div/div[2]")  # div C
                spans = info_div.find_elements(By.XPATH, "./span")
                winrate = spans[0].text.strip() if len(spans) > 0 else ""
                games = spans[1].text.strip().replace("\u202f", "") if len(spans) > 1 else ""

                # -------- lane_quality --------
                lane_quality = ""
                try:
                    button = card.find_element(By.XPATH, "./button")
                    lane_quality = button.text.strip()
                except NoSuchElementException:
                    pass

                # print(f"🎯 {champ_name} | Winrate: {winrate} | Games: {games} | Lane: {lane_quality}")

                results.append({
                    "champ_counter_i": champ_name,
                    "winrate": winrate,
                    "games": games,
                    "lane_quality": lane_quality
                })

            except StaleElementReferenceException:
                print("⚠️ StaleElement — skip")
            except Exception as e:
                print(f"❌ Erreur card {idx} → {e}")
        print(" | ".join(
            f"🎯 {r['champ_counter_i']} {r['winrate']} {r['games']} {r['lane_quality']}"
            for r in results
        ))
    # ===============================
    # 4️⃣ Parser les cards visibles et hors écran
    # ===============================
    parse_cards(visible_wrapper)
    parse_cards(hidden_wrapper)

    print(f"📦 {'Matchups' if matchup else 'Synergies'} collectés : {len(results)}")
    return results


In [15]:
import pandas as pd


def generate_csv_from_champions(
    all_champions_data,
    elo,
    server,
    patch,
    lane_inspected,
    synergy_or_matchup,
    output_path="matchups_champions.csv",
):
    """
    all_champions_data = [
        {
            "name": str,
            "role": str,
            "role_play_ratio": str,
            "tier": str,
            "rank": str,
            "winrate": str,
            "pickrate": str,
            "banrate": str,
            "nb_games_analyzed": str,
            "url": str,
            "collect_coherente": bool,

            "matchups": {
                "top": [ {...}, ... ],
                "jun": [ {...}, ... ],
                "mid": [ {...}, ... ],
                "adc": [ {...}, ... ],
                "sup": [ {...}, ... ],
            },

            "synergies": {
                "top": [ {...}, ... ],
                "jun": [ {...}, ... ],
                "mid": [ {...}, ... ],
                "adc": [ {...}, ... ],
                "sup": [ {...}, ... ],
            }
        },
        ...
    ]
    """

    lanes = ["top", "jun", "mid", "adc", "sup"]

    # ==========================================================
    # 1️⃣ Calculer les max par TYPE (matchup/synergy) et par LANE
    # ==========================================================
    max_matchups = {lane: 0 for lane in lanes}
    max_synergies = {lane: 0 for lane in lanes}

    for champ in all_champions_data:
        for lane in lanes:
            max_matchups[lane] = max(
                max_matchups[lane],
                len(champ.get("matchup", {}).get(lane, []))
            )
            max_synergies[lane] = max(
                max_synergies[lane],
                len(champ.get("synergy", {}).get(lane, []))
            )

    print("📊 Max matchups par lane :", max_matchups)
    print("📊 Max synergies par lane :", max_synergies)

    # ======================
    # 2️⃣ Construire les rows
    # ======================
    rows = []

    for champ in all_champions_data:
        row = {
            # -------- Colonnes principales (EN PREMIER) --------
            "champion": champ.get("name"),
            "role": champ.get("role"),
            "role_play_ratio": champ.get("role_play_ratio"),
            "tier": champ.get("tier"),
            "rank": champ.get("rank"),
            "winrate": champ.get("winrate"),
            "pickrate": champ.get("pickrate"),
            "banrate": champ.get("banrate"),
            "nb_games_analyzed": champ.get("nb_games_analyzed"),
            "url": champ.get("url"),
            "collect_coherente": champ.get("collect_coherente"),
        }

        # ======================
        # 3️⃣ MATCHUPS
        # ======================
        for lane in lanes:
            lane_matchups = champ.get("matchup", {}).get(lane, [])

            for i in range(max_matchups[lane]):
                prefix = f"matchup_{lane}_{i+1}"

                if i < len(lane_matchups):
                    m = lane_matchups[i]
                    row[f"{prefix}_name"] = m.get("champ_counter_i")
                    row[f"{prefix}_winrate"] = m.get("winrate")
                    row[f"{prefix}_games"] = m.get("games")
                    row[f"{prefix}_lane_quality"] = m.get("lane_quality")
                else:
                    row[f"{prefix}_name"] = None
                    row[f"{prefix}_winrate"] = None
                    row[f"{prefix}_games"] = None
                    row[f"{prefix}_lane_quality"] = None

        # ======================
        # 4️⃣ SYNERGIES
        # ======================
        for lane in lanes:
            lane_synergies = champ.get("synergy", {}).get(lane, [])

            for i in range(max_synergies[lane]):
                prefix = f"synergy_{lane}_{i+1}"

                if i < len(lane_synergies):
                    s = lane_synergies[i]
                    row[f"{prefix}_name"] = s.get("champ_counter_i")
                    row[f"{prefix}_winrate"] = s.get("winrate")
                    row[f"{prefix}_games"] = s.get("games")
                    row[f"{prefix}_lane_quality"] = s.get("lane_quality")
                else:
                    row[f"{prefix}_name"] = None
                    row[f"{prefix}_winrate"] = None
                    row[f"{prefix}_games"] = None
                    row[f"{prefix}_lane_quality"] = None

        rows.append(row)

    # ======================
    # 5️⃣ DataFrame + CSV
    # ======================
    df = pd.DataFrame(rows)

    filename = f"{output_path.replace('.csv', '')}{param_lane}_{param_synergy_or_matchup}_{elo}_{server}_{patch}.csv"
    df.to_csv(filename, index=False)

    print(f"✅ CSV généré : {filename}")
    print(f"📐 Shape DF : {df.shape}")


In [16]:
driver = get_or_create_driver()
driver.get("https://dpm.lol/tierlist?tier=gold_plus")

🔁 Tentative de connexion à Chrome existant...


KeyboardInterrupt: 

In [ ]:
# from selenium.webdriver.common.by import By
# from selenium.webdriver.common.action_chains import ActionChains
# from selenium.webdriver.common.keys import Keys
# from selenium.common.exceptions import StaleElementReferenceException
# import time


# def open_champion_in_new_tab(
#     driver,
#     champion_descriptor: dict,
#     wait_seconds: int = 5,
#     param_lane="a definir",
#     param_synergy_or_matchup="a definir"
# ):
#     """
#     champion_descriptor = {
#         "global_index": int,
#         "scroll_y": int,
#         "label": str
#     }
#     """

#     container_selector = (
#         "#root > main > div > div.flex.justify-center.gap-16 > "
#         "div.flex.flex-col.max-w-5xl.w-full.items-center.md\\:items-start.my-8.gap-8.px-12.lg\\:px-0 > "
#         "div.flex.flex-col.w-full.gap-12.bg-black-800.border.border-black-0\\/10.px-16.py-12.rounded-md > "
#         "div.flex.flex-col.gap-y-8.w-full.mb-48 > div:nth-child(3) > div"
#     )

#     scroll_y = champion_descriptor["scroll_y"]
#     label = champion_descriptor["label"]
#     numero= champion_descriptor["numero"]

#     print(f"🧭 Scroll vers Y={scroll_y} pour {label}")
#     driver.execute_script("window.scrollTo(0, arguments[0]);", scroll_y)
#     time.sleep(0.5)  # 👈 important

#     container = driver.find_element(By.CSS_SELECTOR, container_selector)
#     rows = container.find_elements(By.XPATH, "./div/div")

#     if not rows:
#         print("❌ Aucun champion visible après scroll")
#         return

#     row = rows[numero]  # premier visible à cet endroit
#     print(f"🖱️ Ouverture champion → {label}")

#     main_window = driver.current_window_handle

#     ActionChains(driver) \
#         .key_down(Keys.CONTROL) \
#         .click(row) \
#         .key_up(Keys.CONTROL) \
#         .perform()

#     time.sleep(2)

#     windows = driver.window_handles
#     if len(windows) < 2:
#         print("❌ Nouvel onglet non détecté")
#         return

#     new_tab = [w for w in windows if w != main_window][0]

#     for i in range(1, wait_seconds - 1):
#         print(f"⏳ {i}/{wait_seconds}s")
#         time.sleep(1)
#     driver.switch_to.window(new_tab)
#     for i in range(1, wait_seconds):
#         print(f"⏳ {i}/{wait_seconds}s")
#         time.sleep(1)
#     # for i in range(1, wait_seconds):
#     #     print(f"⏳ {i}/{wait_seconds}s")
#     #     time.sleep(1)


#     # dans l'ordre : 
#     # initialiser data_champ 
#     data_champ = { 
#     "name": champion_descriptor["name"],
#     }
#     # Ajouter ces lignes avant d'itérer sur les rôles
#     data_champ["matchups"] = {}   # ✅ initialisation pour éviter KeyError
#     data_champ["synergies"] = {}  # ✅ initialisation pour éviter KeyError
#     # collecter les données du champion dont role_champ, mettre ces données dans le dictionnaire data_champ
#     lane_and_percentage = get_selected_played_lane(driver)
#     # {"lane": None, "percentage": None}
#     data_champ["role"] = lane_and_percentage["lane"]
#     data_champ["role_play_ratio"] = lane_and_percentage["percentage"]
#     general_data_for_champ_at_lane = collect_champion_lane_stats(driver)
#     data_champ["tier"] = general_data_for_champ_at_lane["tier"]
#     data_champ["rank"] = general_data_for_champ_at_lane["rank"]
#     data_champ["winrate"] = general_data_for_champ_at_lane["winrate"]
#     data_champ["pickrate"] = general_data_for_champ_at_lane["pickrate"]
#     data_champ["banrate"] = general_data_for_champ_at_lane["banrate"]
#     data_champ["nb_games_analyzed"] = general_data_for_champ_at_lane["games"]
#     print(data_champ)
#     # cliquer sur matchups
#     click_matchup_or_synergy(driver, matchup=True)
#     roles = ["top", "jun", "mid", "adc", "sup"]
#     # roles = ["top"]

    
#     first_time_matchups = True
#     # for role in roles:
#     for i, role in enumerate(roles):
#         print(i, role)
#         print("lane a inspecter:", param_lane)
#         print("synergy ou matchup:", param_synergy_or_matchup)
#         if not (role == data_champ["role"]):
#             print(f"⚠️ Skipping matchup vs PAS même rôle ({role})")
#             continue
#         print(f"Collecte matchup vs rôle : {role}")
#         # cliquer sur le bouton du rôle
#         click_matchup_or_synergy_lane(driver, role)
#         time.sleep(1)
#         # cliquer sur "liste complète" UNE SEULE FOIS
#         if first_time_matchups:
#             complet = click_liste_complete(driver)
#             print("matchups complets ?", complet)
#             first_time_matchups = False
#             time.sleep(2)
#         # collecter les matchups
#         matchup = collect_matchup(driver, matchup=True)
#         # stocker
#         data_champ["matchups"][role] = matchup
#     # --- SYNERGIES ---
#     # click_matchup_or_synergy(driver, matchup=False)

#     # synergy_roles = [r for r in roles if r != data_champ["role"]]
#     # first_time_synergies = True
#     # for role in synergy_roles:
#     #     print(f"Collecte synergy avec le rôle : {role}")
#     #     # cliquer sur le bouton du rôle
#     #     click_matchup_or_synergy_lane(driver, role)
#     #     time.sleep(1.5)
#     #     # cliquer sur "liste complète" UNE SEULE FOIS
#     #     if first_time_synergies:
#     #         complet = click_liste_complete(driver)
#     #         print("synergies complètes ?", complet)
#     #         first_time_synergies = False
#     #         time.sleep(2)
#     #     # collecter les synergies
#     #     synergy = collect_matchup(driver, matchup=False)
#     #     # stocker
#     #     data_champ["synergies"][role] = synergy

#     print(data_champ)

#     for i in range(1, wait_seconds + 3):
#         print(f"⏳ {i}/{wait_seconds}s")
#         time.sleep(1)
#     print("🌍 URL champion :", driver.current_url)
#     data_champ["url"] = driver.current_url  # (optionnel mais très utile)
#     print("champ name = ", data_champ["name"])
#     data_champ["collect_coherente"] = data_champ["name"].lower() in data_champ["url"].lower()
    
#     driver.close()
#     time.sleep(1)
#     driver.switch_to.window(main_window)
#     print("↩️ Retour liste champions")
#     # print("🌍 URL champion :", driver.current_url)
#     # data_champ["url"] = driver.current_url  # (optionnel mais très utile)
#     time.sleep(5)

#     return data_champ


In [ ]:
# from selenium.webdriver.common.by import By
# from selenium.webdriver.common.action_chains import ActionChains
# from selenium.webdriver.common.keys import Keys
# from selenium.common.exceptions import StaleElementReferenceException
# import time


# def open_champion_in_new_tab(
#     driver,
#     champion_descriptor: dict,
#     wait_seconds: int = 5,
#     param_lane="a definir",
#     param_synergy_or_matchup="a definir"
# ):
#     """
#     champion_descriptor = {
#         "global_index": int,
#         "scroll_y": int,
#         "label": str
#     }
#     """

#     container_selector = (
#         "#root > main > div > div.flex.justify-center.gap-16 > "
#         "div.flex.flex-col.max-w-5xl.w-full.items-center.md\\:items-start.my-8.gap-8.px-12.lg\\:px-0 > "
#         "div.flex.flex-col.w-full.gap-12.bg-black-800.border.border-black-0\\/10.px-16.py-12.rounded-md > "
#         "div.flex.flex-col.gap-y-8.w-full.mb-48 > div:nth-child(3) > div"
#     )

#     scroll_y = champion_descriptor["scroll_y"]
#     label = champion_descriptor["label"]
#     numero= champion_descriptor["numero"]

# # a debugger !!!!!!!!!!!!!!!!!!!!!!!!!!

#     driver.execute_script("window.scrollTo(0, 0);")
#     time.sleep(0.5)
#     print(f"🧭 Scroll vers Y={scroll_y} pour {label}")
#     driver.execute_script("window.scrollTo(0, arguments[0]);", scroll_y)
#     time.sleep(0.5)

#     # position scroll réelle
#     current_scroll = driver.execute_script("return window.scrollY;")
#     print(f"📏 Scroll réel après move = {current_scroll}")

#     container = driver.find_element(By.CSS_SELECTOR, container_selector)
#     rows = container.find_elements(By.XPATH, "./div/div")

#     print(f"📦 Rows trouvées = {len(rows)}")

#     if not rows:
#         print("❌ Aucun champion visible après scroll")
#         return

#     row = rows[numero]

#     # -----------------------------
#     # 📍 INFOS ELEMENT CIBLE
#     # -----------------------------
#     location = row.location
#     size = row.size

#     center_x = location["x"] + size["width"] / 2
#     center_y = location["y"] + size["height"] / 2

#     print("🎯 Element ciblé :")
#     print(f"   label = {label}")
#     print(f"   index = {numero}")
#     print(f"   location = {location}")
#     print(f"   size = {size}")
#     print(f"   center click = ({center_x:.1f}, {center_y:.1f})")

#     # bounding rect JS (plus fiable que Selenium parfois)
#     rect = driver.execute_script("""
#     var r = arguments[0].getBoundingClientRect();
#     return {x:r.x, y:r.y, width:r.width, height:r.height};
#     """, row)

#     print(f"📐 Bounding rect (viewport) = {rect}")

#     print(f"🖱️ Ouverture champion → {label}")

#     main_window = driver.current_window_handle
#     windows_before = driver.window_handles
#     print(f"🪟 Onglets AVANT = {windows_before}")

#     # -----------------------------
#     # CLICK DEBUG
#     # -----------------------------
#     actions = ActionChains(driver)

#     print("⌨️ CTRL down")
#     actions.key_down(Keys.CONTROL)

#     print(f"🖱️ Click ActionChains sur élément centre approx ({center_x:.1f},{center_y:.1f})")
#     actions.click(row)

#     print("⌨️ CTRL up")
#     actions.key_up(Keys.CONTROL)

#     actions.perform()

#     time.sleep(2)

#     # -----------------------------
#     # DEBUG ONGLET OUVERT
#     # -----------------------------
#     windows_after = driver.window_handles
#     print(f"🪟 Onglets APRES = {windows_after}")

#     if len(windows_after) > len(windows_before):
#         new_tab = list(set(windows_after) - set(windows_before))[0]
#         print(f"🆕 Nouvel onglet détecté = {new_tab}")

#         driver.switch_to.window(new_tab)
#         print(f"🌍 URL nouvel onglet = {driver.current_url}")

#         driver.switch_to.window(main_window)
#     else:
#         print("⚠️ Aucun nouvel onglet détecté")


#     # print(f"🧭 Scroll vers Y={scroll_y} pour {label}")
#     # driver.execute_script("window.scrollTo(0, arguments[0]);", scroll_y)
#     # time.sleep(0.5)  # 👈 important

#     # container = driver.find_element(By.CSS_SELECTOR, container_selector)
#     # rows = container.find_elements(By.XPATH, "./div/div")

#     # if not rows:
#     #     print("❌ Aucun champion visible après scroll")
#     #     return

#     # row = rows[numero]  # premier visible à cet endroit
#     # print(f"🖱️ Ouverture champion → {label}")

#     # main_window = driver.current_window_handle

#     # ActionChains(driver) \
#     #     .key_down(Keys.CONTROL) \
#     #     .click(row) \
#     #     .key_up(Keys.CONTROL) \
#     #     .perform()

#     # time.sleep(2)
# # fin a debugger !!!!!!!!!!!!!!!!!!!!!!!!!!

    

#     windows = driver.window_handles
#     if len(windows) < 2:
#         print("❌ Nouvel onglet non détecté")
#         return

#     new_tab = [w for w in windows if w != main_window][0]

#     for i in range(1, wait_seconds - 1):
#         print(f"⏳ {i}/{wait_seconds}s")
#         time.sleep(1)
#     driver.switch_to.window(new_tab)
#     for i in range(1, wait_seconds):
#         print(f"⏳ {i}/{wait_seconds}s")
#         time.sleep(1)
#     # for i in range(1, wait_seconds):
#     #     print(f"⏳ {i}/{wait_seconds}s")
#     #     time.sleep(1)


#     # dans l'ordre : 
#     # initialiser data_champ 
#     data_champ = { 
#     "name": champion_descriptor["name"],
#     }
#     # Ajouter ces lignes avant d'itérer sur les rôles
#     data_champ["matchup"] = {}   # ✅ initialisation pour éviter KeyError
#     data_champ["synergy"] = {}  # ✅ initialisation pour éviter KeyError
#     # collecter les données du champion dont role_champ, mettre ces données dans le dictionnaire data_champ
#     lane_and_percentage = get_selected_played_lane(driver)
#     # {"lane": None, "percentage": None}
#     data_champ["role"] = lane_and_percentage["lane"]
#     data_champ["role_play_ratio"] = lane_and_percentage["percentage"]
#     general_data_for_champ_at_lane = collect_champion_lane_stats(driver)
#     data_champ["tier"] = general_data_for_champ_at_lane["tier"]
#     data_champ["rank"] = general_data_for_champ_at_lane["rank"]
#     data_champ["winrate"] = general_data_for_champ_at_lane["winrate"]
#     data_champ["pickrate"] = general_data_for_champ_at_lane["pickrate"]
#     data_champ["banrate"] = general_data_for_champ_at_lane["banrate"]
#     data_champ["nb_games_analyzed"] = general_data_for_champ_at_lane["games"]
#     print(data_champ)
#     # cliquer sur matchups
#     click_matchup_or_synergy(driver, matchup=(param_synergy_or_matchup == "matchup"))
#     roles = ["top", "jun", "mid", "adc", "sup"]

#     # for role in roles:
#     for i, role in enumerate(roles):
#         print(i, role)
#         print("lane a inspecter:", param_lane)
#         print("synergy ou matchup:", param_synergy_or_matchup)
#         if not (role == param_lane):
#             print(f"⚠️ ⚠️ ⚠️Skipping matchup vs PAS même rôle ({role})")
#             continue
#         if not (param_synergy_or_matchup == "matchup"):
#             if(param_lane == data_champ["role"]):
#                 print(f"⚠️ ⚠️ ⚠️ ⚠️lane analysee est même que lane principale du champion, skipping ({role})")
#                 continue
#         print(f"Collecte matchup vs rôle : {role}")
#         # cliquer sur le bouton du rôle
#         click_matchup_or_synergy_lane(driver, role)
#         time.sleep(1)
#         # cliquer sur "liste complète" UNE SEULE FOIS
#         complet = click_liste_complete(driver)
#         matchup = collect_matchup(driver, matchup=(param_synergy_or_matchup == "matchup"))
#         # stocker
#         colone = param_synergy_or_matchup
#         data_champ[colone][role] = matchup

#     print(data_champ)

#     for i in range(1, wait_seconds - 1):
#         print(f"⏳ {i}/{wait_seconds}s")
#         time.sleep(1)
#     print("🌍 URL champion :", driver.current_url)
#     data_champ["url"] = driver.current_url  # (optionnel mais très utile)
#     print("champ name = ", data_champ["name"])
#     data_champ["collect_coherente"] = data_champ["name"].lower() in data_champ["url"].lower()
    
#     driver.close()
#     time.sleep(1)
#     driver.switch_to.window(main_window)
#     print("↩️ Retour liste champions")
#     # print("🌍 URL champion :", driver.current_url)
#     # data_champ["url"] = driver.current_url  # (optionnel mais très utile)
#     time.sleep(3)

#     return data_champ


In [ ]:
# from selenium.webdriver.common.by import By
# from selenium.webdriver.common.action_chains import ActionChains
# from selenium.webdriver.common.keys import Keys
# from selenium.common.exceptions import StaleElementReferenceException
# import time


# def open_champion_in_new_tab(
#     driver,
#     champion_descriptor: dict,
#     wait_seconds: int = 5,
#     param_lane="a definir",
#     param_synergy_or_matchup="a definir"
# ):
#     """
#     champion_descriptor = {
#         "global_index": int,
#         "scroll_y": int,
#         "label": str
#     }
#     """

#     container_selector = (
#         "#root > main > div > div.flex.justify-center.gap-16 > "
#         "div.flex.flex-col.max-w-5xl.w-full.items-center.md\\:items-start.my-8.gap-8.px-12.lg\\:px-0 > "
#         "div.flex.flex-col.w-full.gap-12.bg-black-800.border.border-black-0\\/10.px-16.py-12.rounded-md > "
#         "div.flex.flex-col.gap-y-8.w-full.mb-48 > div:nth-child(3) > div"
#     )

#     # scroll_y = champion_descriptor["scroll_y"]
#     label = champion_descriptor["label"]
#     numero= champion_descriptor["numero"]

# # a debugger !!!!!!!!!!!!!!!!!!!!!!!!!!

#     print("AAAAAAAAAAAAAA")
#     print(champion_descriptor)
#     print(champion_descriptor)

#     print("scroll top --- ")
#     driver.execute_script("window.scrollTo(0, 0);")
#     time.sleep(0.5)

#     container = driver.find_element(By.CSS_SELECTOR, container_selector)
#     # rows = container.find_elements(By.XPATH, "./div/div")
#     # print("rows avant scroll = ", len(rows))
#     # print(rows)

    
#     print("scroll top container --- ")
#     driver.execute_script("arguments[0].scrollTop = 0;", container_selector)
#     time.sleep(0.5)
#     # scroll page
#     print(f"🧭 Scroll vers Y={champion_descriptor['scroll_page']} pour {label}")
#     driver.execute_script("window.scrollTo(0, arguments[0]);", champion_descriptor["scroll_page"])
#     time.sleep(0.5)
#     # scroll tableau
#     print(f"🧭 Scroll tableau vers Y={champion_descriptor['scroll_table']} pour {label}")
#     driver.execute_script("arguments[0].scrollTop = arguments[1];", container_selector, champion_descriptor["scroll_table"])
#     time.sleep(0.5)
#     time.sleep(0.5)

#     # position scroll réelle
#     current_scroll = driver.execute_script("return window.scrollY;")
#     print(f"📏 Scroll réel après move = {current_scroll}")

#     # container = driver.find_element(By.CSS_SELECTOR, container_selector)
#     # rows = container.find_elements(By.XPATH, "./div/div")

#     rows = container.find_elements(By.XPATH, "./div/div")
#     print("rows trouvées = ", len(rows))
#     print(rows)

#     print(f"📦 Rows trouvées = {len(rows)}")

#     if not rows:
#         print("❌ Aucun champion visible après scroll")
#         return

#     row = rows[numero]

#     # -----------------------------
#     # 📍 INFOS ELEMENT CIBLE
#     # -----------------------------
#     location = row.location
#     size = row.size

#     center_x = location["x"] + size["width"] / 2
#     center_y = location["y"] + size["height"] / 2

#     print("🎯 Element ciblé :")
#     print(f"   label = {label}")
#     print(f"   index = {numero}")
#     print(f"   location = {location}")
#     print(f"   size = {size}")
#     print(f"   center click = ({center_x:.1f}, {center_y:.1f})")

#     # bounding rect JS (plus fiable que Selenium parfois)
#     rect = driver.execute_script("""
#     var r = arguments[0].getBoundingClientRect();
#     return {x:r.x, y:r.y, width:r.width, height:r.height};
#     """, row)

#     print(f"📐 Bounding rect (viewport) = {rect}")

#     print(f"🖱️ Ouverture champion → {label}")

#     main_window = driver.current_window_handle
#     windows_before = driver.window_handles
#     print(f"🪟 Onglets AVANT = {windows_before}")

#     # -----------------------------
#     # CLICK DEBUG
#     # -----------------------------
#     actions = ActionChains(driver)

#     print("⌨️ CTRL down")
#     actions.key_down(Keys.CONTROL)

#     print(f"🖱️ Click ActionChains sur élément centre approx ({center_x:.1f},{center_y:.1f})")
#     actions.click(row)

#     print("⌨️ CTRL up")
#     actions.key_up(Keys.CONTROL)

#     actions.perform()

#     time.sleep(2)

#     # -----------------------------
#     # DEBUG ONGLET OUVERT
#     # -----------------------------
#     windows_after = driver.window_handles
#     print(f"🪟 Onglets APRES = {windows_after}")

#     if len(windows_after) > len(windows_before):
#         new_tab = list(set(windows_after) - set(windows_before))[0]
#         print(f"🆕 Nouvel onglet détecté = {new_tab}")

#         driver.switch_to.window(new_tab)
#         print(f"🌍 URL nouvel onglet = {driver.current_url}")

#         driver.switch_to.window(main_window)
#     else:
#         print("⚠️ Aucun nouvel onglet détecté")


#     # print(f"🧭 Scroll vers Y={scroll_y} pour {label}")
#     # driver.execute_script("window.scrollTo(0, arguments[0]);", scroll_y)
#     # time.sleep(0.5)  # 👈 important

#     # container = driver.find_element(By.CSS_SELECTOR, container_selector)
#     # rows = container.find_elements(By.XPATH, "./div/div")

#     # if not rows:
#     #     print("❌ Aucun champion visible après scroll")
#     #     return

#     # row = rows[numero]  # premier visible à cet endroit
#     # print(f"🖱️ Ouverture champion → {label}")

#     # main_window = driver.current_window_handle

#     # ActionChains(driver) \
#     #     .key_down(Keys.CONTROL) \
#     #     .click(row) \
#     #     .key_up(Keys.CONTROL) \
#     #     .perform()

#     # time.sleep(2)
# # fin a debugger !!!!!!!!!!!!!!!!!!!!!!!!!!

    

#     windows = driver.window_handles
#     if len(windows) < 2:
#         print("❌ Nouvel onglet non détecté")
#         return

#     new_tab = [w for w in windows if w != main_window][0]

#     for i in range(1, wait_seconds - 2):
#         print(f"⏳ {i}/{wait_seconds}s")
#         time.sleep(1)
#     driver.switch_to.window(new_tab)
#     for i in range(1, wait_seconds - 2):
#         print(f"⏳ {i}/{wait_seconds}s")
#         time.sleep(1)
#     # for i in range(1, wait_seconds):
#     #     print(f"⏳ {i}/{wait_seconds}s")
#     #     time.sleep(1)


#     # # dans l'ordre : 
#     # # initialiser data_champ 
#     # data_champ = { 
#     # "name": champion_descriptor["name"],
#     # }
#     # # Ajouter ces lignes avant d'itérer sur les rôles
#     # data_champ["matchup"] = {}   # ✅ initialisation pour éviter KeyError
#     # data_champ["synergy"] = {}  # ✅ initialisation pour éviter KeyError
#     # # collecter les données du champion dont role_champ, mettre ces données dans le dictionnaire data_champ
#     # lane_and_percentage = get_selected_played_lane(driver)
#     # # {"lane": None, "percentage": None}
#     # data_champ["role"] = lane_and_percentage["lane"]
#     # data_champ["role_play_ratio"] = lane_and_percentage["percentage"]
#     # general_data_for_champ_at_lane = collect_champion_lane_stats(driver)
#     # data_champ["tier"] = general_data_for_champ_at_lane["tier"]
#     # data_champ["rank"] = general_data_for_champ_at_lane["rank"]
#     # data_champ["winrate"] = general_data_for_champ_at_lane["winrate"]
#     # data_champ["pickrate"] = general_data_for_champ_at_lane["pickrate"]
#     # data_champ["banrate"] = general_data_for_champ_at_lane["banrate"]
#     # data_champ["nb_games_analyzed"] = general_data_for_champ_at_lane["games"]
#     # print(data_champ)
#     # # cliquer sur matchups
#     # click_matchup_or_synergy(driver, matchup=(param_synergy_or_matchup == "matchup"))
#     # roles = ["top", "jun", "mid", "adc", "sup"]

#     # # for role in roles:
#     # for i, role in enumerate(roles):
#     #     print(i, role)
#     #     print("lane a inspecter:", param_lane)
#     #     print("synergy ou matchup:", param_synergy_or_matchup)
#     #     if not (role == param_lane):
#     #         print(f"⚠️ ⚠️ ⚠️Skipping matchup vs PAS même rôle ({role})")
#     #         continue
#     #     if not (param_synergy_or_matchup == "matchup"):
#     #         if(param_lane == data_champ["role"]):
#     #             print(f"⚠️ ⚠️ ⚠️ ⚠️lane analysee est même que lane principale du champion, skipping ({role})")
#     #             continue
#     #     print(f"Collecte matchup vs rôle : {role}")
#     #     # cliquer sur le bouton du rôle
#     #     click_matchup_or_synergy_lane(driver, role)
#     #     time.sleep(1)
#     #     # cliquer sur "liste complète" UNE SEULE FOIS
#     #     complet = click_liste_complete(driver)
#     #     matchup = collect_matchup(driver, matchup=(param_synergy_or_matchup == "matchup"))
#     #     # stocker
#     #     colone = param_synergy_or_matchup
#     #     data_champ[colone][role] = matchup

#     # print(data_champ)

#     # for i in range(1, wait_seconds - 1):
#     #     print(f"⏳ {i}/{wait_seconds}s")
#     #     time.sleep(1)
#     # print("🌍 URL champion :", driver.current_url)
#     # data_champ["url"] = driver.current_url  # (optionnel mais très utile)
#     # print("champ name = ", data_champ["name"])
#     # data_champ["collect_coherente"] = data_champ["name"].lower() in data_champ["url"].lower()
    
#     time.sleep(1)
#     driver.close()
#     # time.sleep(1)
#     driver.switch_to.window(main_window)
#     print("↩️ Retour liste champions")
#     # print("🌍 URL champion :", driver.current_url)
#     # data_champ["url"] = driver.current_url  # (optionnel mais très utile)
#     time.sleep(3)
#     data_champ = {
#         "name": champion_descriptor["name"],
#     }
#     return data_champ


In [17]:
from selenium.webdriver.common.by import By
import time

def open_champion_in_new_tab(driver, champion_descriptor: dict, wait_seconds: int = 5, param_lane="a definir", param_synergy_or_matchup="a definir"):
    # open_champion_in_new_tab(driver, champ, wait_seconds=5, param_lane=param_lane, param_synergy_or_matchup=param_synergy_or_matchup)
    """
    Ouvre le champion dans un nouvel onglet à partir de l'URL stockée dans champion_descriptor.
    
    champion_descriptor = {
        "name": str,
        "url": str,
        "label": str,
        ...
    }
    """
    url = champion_descriptor.get("url")
    label = champion_descriptor.get("label", "Unknown")

    if not url:
        print(f"❌ Pas d'URL pour {label}, impossible d'ouvrir")
        return None

    print(f"🌍 Ouverture champion '{label}' dans un nouvel onglet → {url}")

    main_window = driver.current_window_handle
    windows_before = driver.window_handles
    print(f"🪟 Onglets AVANT = {windows_before}")

    # Ouvrir nouvel onglet
    driver.execute_script(f"window.open('{url}', '_blank');")
    time.sleep(1)

    windows_after = driver.window_handles
    print(f"🪟 Onglets APRES = {windows_after}")

    # Identifier le nouvel onglet
    new_tabs = [w for w in windows_after if w not in windows_before]
    if not new_tabs:
        print("❌ Aucun nouvel onglet détecté")
        return None

    new_tab = new_tabs[0]
    driver.switch_to.window(new_tab)
    print(f"🔑 Nouvel onglet actif = {new_tab}")
    print(f"📍 URL = {driver.current_url}")

    # Attente optionnelle pour chargement
    for i in range(wait_seconds - 2):
        print(f"⏳ Attente {i+1}/{wait_seconds}s")
        time.sleep(1)




    # dans l'ordre : 
    # initialiser data_champ 
    data_champ = { 
    "name": champion_descriptor["name"],
    }
    # Ajouter ces lignes avant d'itérer sur les rôles
    data_champ["matchup"] = {}   # ✅ initialisation pour éviter KeyError
    data_champ["synergy"] = {}  # ✅ initialisation pour éviter KeyError
    # collecter les données du champion dont role_champ, mettre ces données dans le dictionnaire data_champ
    lane_and_percentage = get_selected_played_lane(driver)
    # {"lane": None, "percentage": None}
    data_champ["role"] = lane_and_percentage["lane"]
    data_champ["role_play_ratio"] = lane_and_percentage["percentage"]
    general_data_for_champ_at_lane = collect_champion_lane_stats(driver)
    data_champ["tier"] = general_data_for_champ_at_lane["tier"]
    data_champ["rank"] = general_data_for_champ_at_lane["rank"]
    data_champ["winrate"] = general_data_for_champ_at_lane["winrate"]
    data_champ["pickrate"] = general_data_for_champ_at_lane["pickrate"]
    data_champ["banrate"] = general_data_for_champ_at_lane["banrate"]
    data_champ["nb_games_analyzed"] = general_data_for_champ_at_lane["games"]
    print(data_champ)
    # cliquer sur matchups
    click_matchup_or_synergy(driver, matchup=(param_synergy_or_matchup == "matchup"))
    roles = ["top", "jun", "mid", "adc", "sup"]

    # for role in roles:
    for i, role in enumerate(roles):
        print(i, role)
        print("lane a inspecter:", param_lane)
        print("synergy ou matchup:", param_synergy_or_matchup)
        if not (role == param_lane):
            print(f"⚠️ ⚠️ ⚠️Skipping matchup vs PAS même rôle ({role})")
            continue
        if not (param_synergy_or_matchup == "matchup"):
            if(param_lane == data_champ["role"]):
                print(f"⚠️ ⚠️ ⚠️ ⚠️lane analysee est même que lane principale du champion, skipping ({role})")
                continue
        print(f"Collecte matchup vs rôle : {role}")
        # cliquer sur le bouton du rôle
        click_matchup_or_synergy_lane(driver, role)
        time.sleep(1)
        # cliquer sur "liste complète" UNE SEULE FOIS
        complet = click_liste_complete(driver)
        matchup = collect_matchup(driver, matchup=(param_synergy_or_matchup == "matchup"))
        # stocker
        colone = param_synergy_or_matchup
        data_champ[colone][role] = matchup

    print(data_champ)

    for i in range(1, wait_seconds - 2):
        print(f"⏳ {i}/{wait_seconds}s")
        time.sleep(1)
    print("🌍 URL champion :", driver.current_url)
    data_champ["url"] = driver.current_url  # (optionnel mais très utile)
    print("champ name = ", data_champ["name"])
    data_champ["collect_coherente"] = data_champ["name"].lower() in data_champ["url"].lower()






    # # Exemple : récupérer juste le nom du champion
    # data_champ = {
    #     "name": champion_descriptor.get("name"),
    #     "url": driver.current_url
    # }

    # Fermer l'onglet et revenir à la fenêtre principale
    driver.close()
    driver.switch_to.window(main_window)
    print("↩️ Retour à la liste des champions")

    return data_champ


In [71]:
driver = get_or_create_driver()
driver.get("https://dpm.lol/tierlist?tier=gold_plus")

🔁 Tentative de connexion à Chrome existant...
✅ Connecté à Chrome existant


In [20]:
from selenium.webdriver.common.by import By
import time

import sys
sys.stdout.flush()

if __name__ == "__main__":

    param_lane = "top"
    param_synergy_or_matchup = "matchup"
    # param_lane = "jun"
    # param_synergy_or_matchup = "matchup"
    # param_lane = "mid"
    # param_synergy_or_matchup = "matchup"
    # param_lane = "adc"
    # param_synergy_or_matchup = "matchup"
    # param_lane = "sup"
    # param_synergy_or_matchup = "matchup"

    # param_lane = "top"
    # param_synergy_or_matchup = "synergy"
    # param_lane = "jun"
    # param_synergy_or_matchup = "synergy"
    # param_lane = "mid"
    # param_synergy_or_matchup = "synergy"
    # param_lane = "adc"
    # param_synergy_or_matchup = "synergy"
    # param_lane = "sup"
    # param_synergy_or_matchup = "synergy"
     
    # Variables pour suivre la combinaison active
    elo = None
    server = None
    patch = None

    filters_container_selector = (
        "#root > main > div > div.flex.justify-center.gap-16 > "
        "div.flex.flex-col.max-w-5xl.w-full.items-center.md\\:items-start.my-8.gap-8.px-12.lg\\:px-0 > "
        "div.flex.flex-col.w-full.gap-12.bg-black-800.border.border-black-0\\/10.px-16.py-12.rounded-md > "
        "div.flex.flex-col.lg\\:flex-row.items-center.justify-between.gap-16.lg\\:gap-24.w-full > "
        "div.flex.flex-row.items-center.justify-center.gap-8.lg\\:gap-16"
    )
    filters_container = driver.find_element(By.CSS_SELECTOR, filters_container_selector)

    print("✅ Containers trouvés")

    # =========================
    # 1️⃣ Ouvrir ELO et noter la liste des boutons
    # =========================
    filters_container.find_element(By.XPATH, ".//button[1]").click()
    time.sleep(0.5)
    elo_buttons = get_last_radix_buttons()
    print(f"\n🎯 ELO détectés : {[b.text for b in elo_buttons]}")

    # =========================
    # 2️⃣ Ouvrir SERVER et noter la liste des boutons
    # =========================
    filters_container.find_element(By.XPATH, ".//button[2]").click()
    time.sleep(0.5)
    server_buttons = get_last_radix_buttons()
    print(f"🌍 SERVER détectés : {[b.text for b in server_buttons]}")

    # =========================
    # 3️⃣ Ouvrir PATCH et noter la liste des boutons
    # =========================
    filters_container.find_element(By.XPATH, ".//div/button").click()
    time.sleep(0.5)
    patch_buttons = get_last_radix_buttons()
    print(f"🧩 PATCH détectés : {[b.text for b in patch_buttons]}")
    

    # =========================
    # BOUCLE SUR TOUTES LES COMBINAISONS
    # =========================


    # de elo_départ à elo_max
    # for i in range(numeloDepart | 0, max(AeloFin, len(elo_buttons))):
    # de elo_départ à elo_fin
    # for i in range(numeloDepart | 0, min(AelohFin, len(elo_buttons))): 
    # for i in range(0,max(1, len(elo_buttons))):
    # for i in range(0,min(2, len(elo_buttons))):
    for i in range(max(1, len(elo_buttons)) - 1, -1, -1):
        if not (i in (0, 1, 3, 5, 7, 9, 11, 13, 15)):
                continue  # 🔹 on skip les serveurs non désirés
        # 🔁 réouvrir la dropdown ELO
        filters_container.find_element(By.XPATH, ".//button[1]").click()
        time.sleep(0.7)
        elo_buttons = get_last_radix_buttons()
        elo_btn = elo_buttons[i]
        elo = elo_btn.text.strip()


        elo_btn.click()
        print(f"\n🎯 ELO [{i}] cliqué → {elo}")
        time.sleep(1)
        print("1")
        time.sleep(1)
        print("2")

        # de server_départ à server_max
        # for j in range(numServerDepart | 0, max(AServerFin, len(server_buttons))):
        # de server_départ à server_fin
        # for j in range(numServerDepart | 0, min(AServerFin, len(server_buttons))): 
        # for j in range(7, min(8, len(server_buttons))):
        #     if ( ((j >= 3) and (j <= 10) and (j != 4) and(j!=7)) ): #7 pour LAS pour les tests


        for j in range(max(8, len(server_buttons)) - 1, -1, -1):
            if ((j >= 3) and (j <= 10)):
                continue  # 🔹 on skip les serveurs non désirés
            # 🔁 réouvrir la dropdown SERVER
            filters_container.find_element(By.XPATH, ".//button[2]").click()
            time.sleep(0.7)
            server_buttons = get_last_radix_buttons()
            server_btn = server_buttons[j]
            server = server_btn.text.strip()


            server_btn.click()
            print(f"  🌍 SERVER [{j}] cliqué → {server}")
            time.sleep(1)
            print("1")
            time.sleep(1)
            print("2")
            time.sleep(1)
            print("3")
            filters_container = driver.find_element(By.CSS_SELECTOR, filters_container_selector)


            # de patch_départ à patch_max
            # for k in range(numPatchDepart | 0, max(APatchFin, len(patch_buttons))):
            # de patch_départ à patch_fin
            # for k in range(numPatchDepart | 0, min(APatchFin, len(patch_buttons))): 
            # for k in range(5, min(6, len(patch_buttons))):  # 🔹 on limite à 3 itérations pour tester          
            for k in range(3, max(5, len(patch_buttons))):  # 🔹 on limite à 3 itérations pour tester          
                # 🔁 réouvrir la dropdown PATCH
                filters_container.find_element(By.XPATH, ".//div/button").click()
                time.sleep(1)

                # 🔹 récupérer à nouveau les boutons PATCH pour éviter StaleElementReference
                patch_buttons = get_last_radix_buttons()
                time.sleep(1)
                print("click sur les patchs")
                print(f"🧩 PATCH mis à jour : {[b.text for b in patch_buttons]}")
                patch_btn = patch_buttons[k]

                # 🔹 cliquer sur le kème bouton
                patch = patch_btn.text.strip()
                patch_btn.click()
                time.sleep(0.7)

                print(f"    🧩 PATCH [{k}] cliqué → {patch}")

                # ✅ COMBINAISON ACTIVE
                print(f"    ✅ COMBINAISON ACTIVE : ELO={elo}, SERVER={server}, PATCH={patch}")
                time.sleep(1)
                print("1")
                time.sleep(1)
                print("2")
                time.sleep(1)
                print("3")
                
                filters_container = driver.find_element(By.CSS_SELECTOR, filters_container_selector)
                

                champions = init_parse(driver)
                all_champions_data = []  # liste qui va contenir les objets champion
                for l, champ in enumerate(champions):
                # for l, champ in enumerate(champions[12:15], start=12):
                    print(champ)
                    champ_data = open_champion_in_new_tab(driver, champ, wait_seconds=5, param_lane=param_lane, param_synergy_or_matchup=param_synergy_or_matchup)
                    all_champions_data.append(champ_data)


                    if (l % 5 == 0) and (l != 0):  # tous les 5 champions, on sauvegarde dans un CSV intermédiaire pour éviter de tout perdre en cas de bug
                        generate_csv_from_champions(
                            all_champions_data,
                            elo=elo,
                            server=server,
                            patch=patch,
                            lane_inspected=param_lane,
                            synergy_or_matchup=param_synergy_or_matchup,
                            output_path=f"{l}_champs_.csv"
                        )    
                        all_champions_data = []
                generate_csv_from_champions(
                    all_champions_data,
                    elo=elo,
                    server=server,
                    patch=patch,
                    lane_inspected=param_lane,
                    synergy_or_matchup=param_synergy_or_matchup,
                    output_path=f"end_champs_.csv"
                )  

                

                driver.execute_script("window.scrollTo(0, arguments[0]);", 0)




[Span 4] → 'rang'
[Span 5] → '52.0
%
winrate'
✅ winrate détecté → '52.0 %'
[Span 6] → '%'
[Span 7] → 'winrate'
[Span 8] → '3.8
%
pickrate'
✅ pickrate détecté → '3.8 %'
[Span 9] → '%'
[Span 10] → 'pickrate'
[Span 11] → '5.3
%
banrate'
✅ banrate détecté → '5.3 %'
[Span 12] → '%'
[Span 13] → 'banrate'
[Span 14] → '334 836
parties'
✅ games détecté → '334836'
[Span 15] → 'parties'
------------------------------------------------------------
✅ Stats collectées : {'tier': 'A', 'rank': '10 / 44', 'winrate': '52.0 %', 'pickrate': '3.8 %', 'banrate': '5.3 %', 'games': '334836'}
{'name': 'Xerath', 'matchup': {}, 'synergy': {}, 'role': 'mid', 'role_play_ratio': '55.6%', 'tier': 'A', 'rank': '10 / 44', 'winrate': '52.0 %', 'pickrate': '3.8 %', 'banrate': '5.3 %', 'nb_games_analyzed': '334836'}
🟢 Sélection de l'onglet MATCHUPS
🔍 MATCHUPS data-state = active
ℹ️ Onglet MATCHUPS déjà actif
0 top
lane a inspecter: top
synergy ou matchup: matchup
Collecte matchup vs rôle : top
🟢 Sélection de la lane top 

KeyboardInterrupt: 

In [19]:
driver.get("https://dpm.lol/tierlist?tier=gold_plus")

In [18]:
# driver = get_or_create_driver()
# driver.get("https://dpm.lol/tierlist?tier=gold_plus")

options = webdriver.ChromeOptions()
options.add_argument(f"--remote-debugging-port={DEBUG_PORT}")
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

In [2]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import os, time

def test_worker(task):
    print(f"[TID {os.getpid()}] WORKER START → {task}", flush=True)
    time.sleep(2)
    print(f"[TID {os.getpid()}] WORKER END → {task}", flush=True)
    return f"Done {task['role']}"

tasks = [{"role": "top"}, {"role": "mid"}, {"role": "adc"}]

results = []
with ThreadPoolExecutor(max_workers=3) as executor:
    futures = [executor.submit(test_worker, task) for task in tasks]
    for future in as_completed(futures):
        results.append(future.result())

print("RESULTS:", results)


[TID 4820] WORKER START → {'role': 'top'}


[TID 4820] WORKER START → {'role': 'mid'}
[TID 4820] WORKER START → {'role': 'adc'}
[TID 4820] WORKER END → {'role': 'adc'}[TID 4820] WORKER END → {'role': 'top'}
[TID 4820] WORKER END → {'role': 'mid'}

RESULTS: ['Done top', 'Done adc', 'Done mid']


In [21]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import time, os
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

def browser_worker(task):
    print(f"[PID {os.getpid()}] WORKER START → {task['name']}", flush=True)
    # Options Chrome (ne pas activer headless)
    options = Options()
    options.add_argument("--start-maximized")
    # options.add_argument("--disable-gpu")  # pas nécessaire mais possible
    # Crée le service avec webdriver-manager (installe le chromedriver si besoin)
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    try:
        driver.get(task["url"])
        print(f"[PID {os.getpid()}] {task['name']} opened {task['url']}", flush=True)
        # Laisser la fenêtre ouverte pendant la durée demandée
        time.sleep(task.get("duration", 10))
    finally:
        driver.quit()
        print(f"[PID {os.getpid()}] WORKER END → {task['name']}", flush=True)
    return f"Done {task['name']}"

if __name__ == "__main__":
    tasks = [
        {"name": "top", "url": "https://www.google.com", "duration": 8},
        {"name": "mid", "url": "https://www.python.org", "duration": 10},
        {"name": "adc", "url": "https://www.github.com", "duration": 12},
    ]

    results = []
    with ThreadPoolExecutor(max_workers=3) as executor:
        futures = [executor.submit(browser_worker, task) for task in tasks]
        for future in as_completed(futures):
            results.append(future.result())

    print("RESULTS:", results)


[PID 46452] WORKER START → top
[PID 46452] WORKER START → mid
[PID 46452] WORKER START → adc


[PID 46452] mid opened https://www.python.org
[PID 46452] top opened https://www.google.com
[PID 46452] adc opened https://www.github.com
[PID 46452] WORKER END → top
[PID 46452] WORKER END → mid
[PID 46452] WORKER END → adc
RESULTS: ['Done top', 'Done mid', 'Done adc']


In [24]:
import pandas as pd

pd.set_option("display.max_columns", None)


# chemin vers ton CSV
csv_path = "matchups_champions_Challenger_LAS_16.1.csv"  # adapte si besoin

# ouverture du CSV dans un DataFrame
df = pd.read_csv(csv_path)

# aperçu rapide
df.head()


,champion,role,role_play_ratio,tier,rank,winrate,pickrate,banrate,nb_games_analyzed,matchup_top_1_name,matchup_top_1_winrate,matchup_top_1_games,matchup_top_1_lane_quality,matchup_top_2_name,matchup_top_2_winrate,matchup_top_2_games,matchup_top_2_lane_quality,matchup_top_3_name,matchup_top_3_winrate,matchup_top_3_games,matchup_top_3_lane_quality,matchup_top_4_name,matchup_top_4_winrate,matchup_top_4_games,matchup_top_4_lane_quality,matchup_top_5_name,matchup_top_5_winrate,matchup_top_5_games,matchup_top_5_lane_quality,matchup_top_6_name,matchup_top_6_winrate,matchup_top_6_games,matchup_top_6_lane_quality,matchup_top_7_name,matchup_top_7_winrate,matchup_top_7_games,matchup_top_7_lane_quality,matchup_top_8_name,matchup_top_8_winrate,matchup_top_8_games,matchup_top_8_lane_quality,matchup_top_9_name,matchup_top_9_winrate,matchup_top_9_games,matchup_top_9_lane_quality,matchup_top_10_name,matchup_top_10_winrate,matchup_top_10_games,matchup_top_10_lane_quality,matchup_top_11_name,matchup_top_11_winrate,matchup_top_11_games,matchup_top_11_lane_quality,matchup_top_12_name,matchup_top_12_winrate,matchup_top_12_games,matchup_top_12_lane_quality,matchup_top_13_name,matchup_top_13_winrate,matchup_top_13_games,matchup_top_13_lane_quality,matchup_top_14_name,matchup_top_14_winrate,matchup_top_14_games,matchup_top_14_lane_quality,matchup_top_15_name,matchup_top_15_winrate,matchup_top_15_games,matchup_top_15_lane_quality,matchup_top_16_name,matchup_top_16_winrate,matchup_top_16_games,matchup_top_16_lane_quality,matchup_top_17_name,matchup_top_17_winrate,matchup_top_17_games,matchup_top_17_lane_quality,matchup_top_18_name,matchup_top_18_winrate,matchup_top_18_games,matchup_top_18_lane_quality,matchup_top_19_name,matchup_top_19_winrate,matchup_top_19_games,matchup_top_19_lane_quality,matchup_top_20_name,matchup_top_20_winrate,matchup_top_20_games,matchup_top_20_lane_quality,matchup_top_21_name,matchup_top_21_winrate,matchup_top_21_games,matchup_top_21_lane_quality,matchup_top_22_name,matchup_top_22_winrate,matchup_top_22_games,matchup_top_22_lane_quality,matchup_top_23_name,matchup_top_23_winrate,matchup_top_23_games,matchup_top_23_lane_quality,matchup_top_24_name,matchup_top_24_winrate,matchup_top_24_games,matchup_top_24_lane_quality,matchup_top_25_name,matchup_top_25_winrate,matchup_top_25_games,matchup_top_25_lane_quality,matchup_top_26_name,matchup_top_26_winrate,matchup_top_26_games,matchup_top_26_lane_quality,matchup_top_27_name,matchup_top_27_winrate,matchup_top_27_games,matchup_top_27_lane_quality,matchup_top_28_name,matchup_top_28_winrate,matchup_top_28_games,matchup_top_28_lane_quality,matchup_top_29_name,matchup_top_29_winrate,matchup_top_29_games,matchup_top_29_lane_quality,matchup_top_30_name,matchup_top_30_winrate,matchup_top_30_games,matchup_top_30_lane_quality,matchup_top_31_name,matchup_top_31_winrate,matchup_top_31_games,matchup_top_31_lane_quality,matchup_top_32_name,matchup_top_32_winrate,matchup_top_32_games,matchup_top_32_lane_quality,matchup_top_33_name,matchup_top_33_winrate,matchup_top_33_games,matchup_top_33_lane_quality,matchup_top_34_name,matchup_top_34_winrate,matchup_top_34_games,matchup_top_34_lane_quality,matchup_top_35_name,matchup_top_35_winrate,matchup_top_35_games,matchup_top_35_lane_quality,matchup_top_36_name,matchup_top_36_winrate,matchup_top_36_games,matchup_top_36_lane_quality,matchup_top_37_name,matchup_top_37_winrate,matchup_top_37_games,matchup_top_37_lane_quality,matchup_top_38_name,matchup_top_38_winrate,matchup_top_38_games,matchup_top_38_lane_quality,matchup_top_39_name,matchup_top_39_winrate,matchup_top_39_games,matchup_top_39_lane_quality,matchup_top_40_name,matchup_top_40_winrate,matchup_top_40_games,matchup_top_40_lane_quality,matchup_top_41_name,matchup_top_41_winrate,matchup_top_41_games,matchup_top_41_lane_quality,matchup_top_42_name,matchup_top_42_winrate,matchup_top_42_games,matchup_top_42_lane_quality,matchup_top_43_name,matchup_top_43_winrate,matchup_top_43_games,matchup_top_43_lane_qual

In [21]:
df.columns.tolist()

['champion',
 'role',
 'role_play_ratio',
 'tier',
 'rank',
 'winrate',
 'pickrate',
 'banrate',
 'nb_games_analyzed',
 'matchup_top_1_name',
 'matchup_top_1_winrate',
 'matchup_top_1_games',
 'matchup_top_1_lane_quality',
 'matchup_top_2_name',
 'matchup_top_2_winrate',
 'matchup_top_2_games',
 'matchup_top_2_lane_quality',
 'matchup_top_3_name',
 'matchup_top_3_winrate',
 'matchup_top_3_games',
 'matchup_top_3_lane_quality',
 'matchup_top_4_name',
 'matchup_top_4_winrate',
 'matchup_top_4_games',
 'matchup_top_4_lane_quality',
 'matchup_top_5_name',
 'matchup_top_5_winrate',
 'matchup_top_5_games',
 'matchup_top_5_lane_quality',
 'matchup_top_6_name',
 'matchup_top_6_winrate',
 'matchup_top_6_games',
 'matchup_top_6_lane_quality',
 'matchup_top_7_name',
 'matchup_top_7_winrate',
 'matchup_top_7_games',
 'matchup_top_7_lane_quality',
 'matchup_top_8_name',
 'matchup_top_8_winrate',
 'matchup_top_8_games',
 'matchup_top_8_lane_quality',
 'matchup_top_9_name',
 'matchup_top_9_winrate',


In [22]:
pd.set_option("display.max_colwidth", None)
df.iloc[0]

champion                       Aphelios
role                                adc
role_play_ratio                   99.6%
tier                                 S+
rank                              1 / 2
                                 ...   
synergy_sup_27_lane_quality         NaN
synergy_sup_28_name               Yuumi
synergy_sup_28_winrate            0.00%
synergy_sup_28_games                  1
synergy_sup_28_lane_quality         NaN
Name: 0, Length: 1521, dtype: object